### eNextSUT generator

Run this code to generate (or update) our eNextSUT reference database.
By default, it relies on Exiobase Hybrid version (v3.3.18) but could be potentially arranged to any other SUT in the future. 

As if 14th October 2024, the above-mentioned database is parsed, aggregated in terms of electricity commodity (1 commodity) and activities (EMBER power plants technologies) and electricity production mixes are updated according to EMBER ones for a given year.

Just mind to update your paths in the 'paths.yml' file and to specify the user (your initials) and the desired electricity mix in the first cell, then run all the code

In [1]:
import mario
import yaml
import pandas as pd

user = 'LR'   # change this to your username
year = 2022   # change this to the year you want to update the electricity mixes to

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

cvxpy module is not installed in your system. This will raise problems in some of the abilities of MARIO


In [2]:
# Parse raw SUT
world = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

{'Y': Region                                                                                  AT  \
Level                                                                 Consumption category   
Item                                                                Changes in inventories   
Region Level     Item                                                                        
AT     Activity  Activities auxiliary to financial intermediatio...               0.000000   
                 Activities of membership organisation n.e.c. (91)                0.000000   
                 Air transport (62)                                               0.000000   
                 Aluminium production                                             0.000000   
                 Animal products nec                                              0.000000   
...                                                                                    ...   
ZA     Commodity Wood material for treatment; Re-proce

In [3]:
# Aggregate electricity commodities and activities to match EMBER
world.aggregate("support data/aggregate_ee.xlsx",ignore_nan=True)

nan values for the aggregation of Activity for following items ignored
['Cultivation of paddy rice', 'Cultivation of wheat', 'Cultivation of cereal grains nec', 'Cultivation of vegetables, fruit, nuts', 'Cultivation of oil seeds', 'Cultivation of sugar cane, sugar beet', 'Cultivation of plant-based fibers', 'Cultivation of crops nec', 'Cattle farming', 'Pigs farming', 'Poultry farming', 'Meat animals nec', 'Animal products nec', 'Raw milk', 'Wool, silk-worm cocoons', 'Manure treatment (conventional), storage and land application', 'Manure treatment (biogas), storage and land application', 'Forestry, logging and related service activities (02)', 'Fishing, operating of fish hatcheries and fish farms; service activities incidental to fishing (05)', 'Mining of coal and lignite; extraction of peat (10)', 'Extraction of crude petroleum and services related to crude oil extraction, excluding surveying', 'Extraction of natural gas and services related to natural gas extraction, excluding surve

In [4]:
# Parse electricity mixes data
ee_mixes = pd.read_excel("support data/EMBER_EXIOBASE mixes.xlsx" ,sheet_name=str(year),index_col=[0])

In [5]:
z = world.z
s = world.s

for region in world.get_index('Region'):
    new_mix = ee_mixes.loc[region,:].to_frame().sort_index(level=0) 

    s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')] = new_mix.values  # check if commodity electricity is called "Electricity" in aggregation excel file
    
z.update(s)

world.update_scenarios('baseline',z=z)
world.reset_to_coefficients('baseline')


In [ ]:
world.to_txt(paths['export'])

Database: to calculate V following matrices are need.
['X'].Trying to calculate dependencies.
